In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd

In [3]:
df_covid = pd.read_excel('data/covid_filtered.xlsx')
df_covid.head()

,EVOLUCI,FECDEF,DEFVERIFI,RESDEFIN,ANTIGENCOVID,CLASCOVID19,ENTRESI
0,SEGUIMIENTO TERMINADO,NaT,NaN,NO POSITIVO A SARS-COV-2,NaN,NEGATIVO,BAJA CALIFORNIA SUR
1,DEFUNCION,2020-11-10,SI,POSITIVO A SARS-COV-2,NaN,CONFIRMADO,BAJA CALIFORNIA SUR
2,SEGUIMIENTO TERMINADO,NaT,NaN,POSITIVO A SARS-COV-2,NaN,CONFIRMADO,BAJA CALIFORNIA SUR
3,SEGUIMIENTO TERMINADO,NaT,NaN,NO POSITIVO A SARS-COV-2,NaN,NEGATIVO,BAJA CALIFORNIA SUR
4,SEGUIMIENTO TERMINADO,NaT,NaN,NO POSITIVO A SARS-COV-2,NaN,NEGATIVO,BAJA CALIFORNIA SUR


In [4]:
covid_positives = df_covid[
    (df_covid['RESDEFIN'].isin([
        'POSITIVO A SARS-COV-2', 
        'SARS-COV-2 - ALPHA', 
        'SARS-COV-2 - LAMBDA', 
        'SARS-COV-2 - DELTA'
    ])) |
    (df_covid['CLASCOVID19'].isin([
        'CONFIRMADO', 
        'CONF ASO', 
        'CONF DIC'
    ])) |
    (df_covid['ANTIGENCOVID'] == 'POSITIVO A SARS-COV-2')
]

covid_deaths = df_covid[
    (df_covid['EVOLUCI'] == 'DEFUNCION') | 
    (df_covid['FECDEF'].notna()) |
    (df_covid['DEFVERIFI'] == 'SI')  
]


# Agrupar por estado (ENTRESI)
covid_positives_by_state = covid_positives.groupby('ENTRESI').size().reset_index(name='CASOS')
covid_deaths_by_state = covid_deaths.groupby('ENTRESI').size().reset_index(name='DEFUNCIONES')

covid = covid_positives_by_state.merge(covid_deaths_by_state, on='ENTRESI', how='left')
covid

,ENTRESI,CASOS,DEFUNCIONES
0,AGUASCALIENTES,2357,289
1,BAJA CALIFORNIA,5589,1266
2,BAJA CALIFORNIA SUR,2517,177
3,CAMPECHE,1055,196
4,CHIAPAS,1187,275
5,CHIHUAHUA,5387,1047
6,CIUDAD DE MEXICO,52353,4376
7,COAHUILA DE ZARAGOZA,7415,931
8,COLIMA,1170,172
9,DURANGO,3795,343


Se utilizo el geoJSON de https://github.com/angelnmara/geojson/blob/master/mexicoHigh.json para obtener los centroides de los estados.

In [5]:
# Poblacion por estado
population = {
    'AGUASCALIENTES': 1425607,
    'BAJA CALIFORNIA': 3769020,
    'BAJA CALIFORNIA SUR': 798447,
    'CAMPECHE': 928363,
    'COAHUILA DE ZARAGOZA': 3146771,
    'COLIMA': 731391,
    'CHIAPAS': 5543828,
    'CHIHUAHUA': 3741869,
    'CIUDAD DE MEXICO': 9209944,
    'DURANGO': 1832650,
    'GUANAJUATO': 6166934,
    'GUERRERO': 3540685,
    'HIDALGO': 3082841,
    'JALISCO': 8348151,
    'MEXICO': 16992418,
    'MICHOACAN DE OCAMPO': 4748846,
    'MORELOS': 1971520,
    'NAYARIT': 1235456,
    'NUEVO LEON': 5784442,
    'OAXACA': 4132148,
    'PUEBLA': 6583278,
    'QUERETARO': 2368467,
    'QUINTANA ROO': 1857985,
    'SAN LUIS POTOSI': 2822255,
    'SINALOA': 3026943,
    'SONORA': 2944840,
    'TABASCO': 2402598,
    'TAMAULIPAS': 3527735,
    'TLAXCALA': 1342977,
    'VERACRUZ DE IGNACIO DE LA LLAVE': 8062579,
    'YUCATAN': 2320898,
    'ZACATECAS': 1622138
}

# Crear un diccionario para mapear los nombres de los estados
states_map = {
    'Aguascalientes': 'AGUASCALIENTES',
    'Baja California': 'BAJA CALIFORNIA',
    'Baja California Sur': 'BAJA CALIFORNIA SUR',
    'Campeche': 'CAMPECHE',
    'Coahuila': 'COAHUILA DE ZARAGOZA',
    'Colima': 'COLIMA',
    'Chiapas': 'CHIAPAS',
    'Chihuahua': 'CHIHUAHUA',
    'Ciudad de México': 'CIUDAD DE MEXICO',
    'Durango': 'DURANGO',
    'Guanajuato': 'GUANAJUATO',
    'Guerrero': 'GUERRERO',
    'Hidalgo': 'HIDALGO',
    'Jalisco': 'JALISCO',
    'México': 'MEXICO',
    'Michoacán': 'MICHOACAN DE OCAMPO',
    'Morelos': 'MORELOS',
    'Nayarit': 'NAYARIT',
    'Nuevo León': 'NUEVO LEON',
    'Oaxaca': 'OAXACA',
    'Puebla': 'PUEBLA',
    'Querétaro': 'QUERETARO',
    'Quintana Roo': 'QUINTANA ROO',
    'San Luis Potosí': 'SAN LUIS POTOSI',
    'Sinaloa': 'SINALOA',
    'Sonora': 'SONORA',
    'Tabasco': 'TABASCO',
    'Tamaulipas': 'TAMAULIPAS',
    'Tlaxcala': 'TLAXCALA',
    'Veracruz': 'VERACRUZ DE IGNACIO DE LA LLAVE',
    'Yucatán': 'YUCATAN',
    'Zacatecas': 'ZACATECAS'
}

# Añadir la población al DataFrame final
covid['POB'] = covid['ENTRESI'].map(population)
covid

,ENTRESI,CASOS,DEFUNCIONES,POB
0,AGUASCALIENTES,2357,289,1425607
1,BAJA CALIFORNIA,5589,1266,3769020
2,BAJA CALIFORNIA SUR,2517,177,798447
3,CAMPECHE,1055,196,928363
4,CHIAPAS,1187,275,5543828
5,CHIHUAHUA,5387,1047,3741869
6,CIUDAD DE MEXICO,52353,4376,9209944
7,COAHUILA DE ZARAGOZA,7415,931,3146771
8,COLIMA,1170,172,731391
9,DURANGO,3795,343,1832650


In [6]:
covid.to_csv('./data/covid_clean.csv', index=False)